In [3]:
import pandas as pd
import numpy as np

from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score,
    adjusted_rand_score
)

In [4]:
X = pd.read_csv("../data/clustering_features.csv")
X_scaled = np.load("../data/X_scaled.npy")

print(X.shape)
print(X_scaled.shape)

(3900, 8)
(3900, 8)


In [5]:
reference_model = KMeans(
    n_clusters=2,
    random_state=42,
    n_init=10
)

reference_labels = reference_model.fit_predict(X_scaled)

reference_silhouette = silhouette_score(
    X_scaled,
    reference_labels
)

print("Reference silhouette:", reference_silhouette)

C:\Users\21626\AppData\Roaming\Python\Python38\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] Le fichier spécifié est introuvable
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\21626\AppData\Roaming\Python\Python38\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
  File "c:\Program Files\Python38\lib\subprocess.py", line 489, in run
    with Popen(*popenargs, **kwargs) as process:
  File "c:\Program Files\Python38\lib\subprocess.py", line 854, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "c:\Program Files\Python38\lib\subprocess.py", line 1307, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,


Reference silhouette: 0.21232643227860026


In [6]:
stability_results = []

for seed in range(10):
    model = KMeans(
        n_clusters=2,
        random_state=seed,
        n_init=10
    )
    
    labels = model.fit_predict(X_scaled)
    
    stability_results.append({
        "seed": seed,
        "silhouette": silhouette_score(
            X_scaled,
            labels
        ),
        "ARI_vs_reference": adjusted_rand_score(
            reference_labels,
            labels
        )
    })

stability_df = pd.DataFrame(stability_results)

stability_df

,seed,silhouette,ARI_vs_reference
0,0,0.212339,0.996925
1,1,0.212339,0.996925
2,2,0.212336,0.996925
3,3,0.212326,1.000000
4,4,0.212339,0.996925
5,5,0.212322,0.996925
6,6,0.212322,0.996925
7,7,0.212332,0.997949
8,8,0.212322,0.997949
9,9,0.212323,0.998974


In [7]:
profile = X.copy()
profile["Cluster"] = reference_labels

cluster_summary = profile.groupby("Cluster").agg(
    ["mean", "median", "std"]
)

cluster_summary

Age                   Purchase Amount (USD)                    \
              mean median        std                  mean median        std   
Cluster                                                                        
0        44.199396   44.0  15.329447             58.439577   58.0  24.021917   
1        43.932602   44.0  15.082909             61.138976   62.0  23.257676   

        Review Rating                  Previous Purchases  ...  \
                 mean median       std               mean  ...   
Cluster                                                    ...   
0            3.745871    3.7  0.718665          23.891239  ...   
1            3.754180    3.8  0.713843          26.866771  ...   

        Purchase Frequency Is_Subscribed                  Log_Annual_Spend  \
                       std          mean median       std             mean   
Cluster                                                                      
0                 3.965159      0.263847    0.0  0.440828         5.122712   
1                14.011288      0.276385    0.0  0.447326         7.356585   

                           Log_Purchase_Engagement                      
          median       std                    mean    median       std  
Cluster                                                                 
0        5.26269  0.885563                4.038541  4.174387  1.081385  
1        7.41758  0.606205                6.392698  6.478510  0.864729  

[2 rows x 24 columns]

In [8]:
from sklearn.model_selection import train_test_split
robustness_results = []

for seed in range(10):

    # Sélection aléatoire de 80% des clients
    indices, _ = train_test_split(
        np.arange(len(X_scaled)),
        test_size=0.20,
        random_state=seed
    )

    # Données du sous-échantillon
    X_sample = X_scaled[indices]

    # Référence correspondante
    reference_sample_labels = reference_labels[indices]

    # Nouveau K-Means
    model = KMeans(
        n_clusters=2,
        random_state=seed,
        n_init=10
    )

    sample_labels = model.fit_predict(X_sample)

    # Comparaison avec la segmentation de référence
    ari = adjusted_rand_score(
        reference_sample_labels,
        sample_labels
    )

    robustness_results.append({
        "seed": seed,
        "sample_size": len(indices),
        "ARI_vs_reference": ari
    })

robustness_df = pd.DataFrame(robustness_results)

robustness_df

,seed,sample_size,ARI_vs_reference
0,0,3120,0.996156
1,1,3120,0.985943
2,2,3120,0.996156
3,3,3120,0.989767
4,4,3120,0.996156
5,5,3120,0.984670
6,6,3120,0.965673
7,7,3120,0.982126
8,8,3120,0.993598
9,9,3120,0.988491


In [9]:
np.save(
    "../data/reference_labels.npy",
    reference_labels
)

print("Reference labels exported:", reference_labels.shape)

Reference labels exported: (3900,)
